In [1]:
import pandas as pd
import numpy as np
import pickle

In [2]:
from grape import Graph

### Preprocess and assign IDs to NP-KG + train-test split for embedding experiments

In [3]:
KG_PATH = '/ix/rboyce/sbt12/np-kg/resources/knowledge_graphs/'
DIR_PATH = '../data/'

In [4]:
KG_FILE = 'NP-KG_BETWEENNESS-WEIGHTED-PK-CURIE-HASHED_v3.0.0.tsv'

In [5]:
npkg_hashed = Graph.from_csv(
    node_path=KG_PATH+'nodeTypes_CURIE_v3.0.0.tsv',
    node_list_node_types_column_number=1,
    nodes_column_number=0,
    node_list_separator='\t',
    node_list_header=True,
    edge_path=KG_PATH+KG_FILE,
    edge_list_separator='\t',
    edge_list_header=True,
    edge_list_edge_types_column_number=1,
    sources_column_number=0,
    destinations_column_number=2,
    weights_column_number=3,
    directed=True,
    verbose=True
)

In [6]:
npkg_hashed.get_number_of_connected_components()

(348, 1, 1089125)

In [7]:
npkg_hashed = npkg_hashed.remove_disconnected_nodes()

In [8]:
npkg_hashed.get_number_of_edges()

7786923

In [6]:
###dump all edges to file for embeddings
npkg_hashed.dump_edges(
    DIR_PATH+'npkg/train.csv',
    separator=',',
    header = False,
    sources_column_number = 0,
    destinations_column_number = 2,
    edge_types_column_number = 1,
    numeric_node_ids = False,
    directed = True
)

In [22]:
npkg_hashed.get_number_of_edges()

7786923

In [ ]:
npkg_df = pd.DataFrame(columns=['subject', 'relation', 'object'])

In [25]:
npkg_hashed.get_edge_weight_from_edge_id(0)

1021.6398315429688

### Split into train, validation and test

In [9]:
train, test = npkg_hashed.connected_holdout(
    train_size = 0.8,
    random_state=42,
    include_all_edge_types=True,
    verbose=True
)

In [10]:
print('Train holdout:')
print('Number of edges: ', train.get_number_of_edges())
print('Number of nodes: ', train.get_number_of_nodes())
print('Edge types: ', train.get_number_of_edge_types())

Train holdout:
Number of edges:  6229539
Number of nodes:  1089125
Edge types:  1343


In [11]:
print('Test holdout:')
print('Number of edges: ', test.get_number_of_edges())
print('Number of nodes: ', test.get_number_of_nodes())
print('Edge types: ', test.get_number_of_edge_types())

Test holdout:
Number of edges:  1557384
Number of nodes:  1089125
Edge types:  1343


In [12]:
train_sub, valid = train.connected_holdout(
    train_size = 0.9,
    random_state=42,
    include_all_edge_types=True,
    verbose=True
)

In [13]:
print('Train holdout:')
print('Number of edges: ', train_sub.get_number_of_edges())
print('Number of nodes: ', train_sub.get_number_of_nodes())
print('Edge types: ', train_sub.get_number_of_edge_types())

Train holdout:
Number of edges:  5606586
Number of nodes:  1089125
Edge types:  1343


In [14]:
print('Validation holdout:')
print('Number of edges: ', valid.get_number_of_edges())
print('Number of nodes: ', valid.get_number_of_nodes())
print('Edge types: ', valid.get_number_of_edge_types())

Validation holdout:
Number of edges:  622953
Number of nodes:  1089125
Edge types:  1343


### Dump nodes and edges with IDs

In [15]:
node_mapping = npkg_hashed.get_nodes_mapping()
with open(DIR_PATH+'node_mapping.pickle', 'wb') as fileo:
    pickle.dump(node_mapping, fileo)

In [16]:
##N=5606586 (70%)
train_sub.dump_edges(
    DIR_PATH+'train.csv',
    separator=',',
    header = False,
    sources_column_number = 0,
    destinations_column_number = 2,
    edge_types_column_number = 1,
    numeric_node_ids = False,
    directed = True
)

In [17]:
##N=622953 (10%)
valid.dump_edges(
    DIR_PATH+'valid.csv',
    separator=',',
    header = False,
    sources_column_number = 0,
    destinations_column_number = 2,
    edge_types_column_number = 1,
    numeric_node_ids = False,
    directed = True
)

In [18]:
##N=1557384 (20%)
test.dump_edges(
    DIR_PATH+'test.csv',
    separator=',',
    header = False,
    sources_column_number = 0,
    destinations_column_number = 2,
    edge_types_column_number = 1,
    numeric_node_ids = False,
    directed = True
)

### Load data and confirm format

In [19]:
npkg_df = pd.read_csv(DIR_PATH+'npkg/train.csv', header=None, names=['subject', 'predicate', 'object'])
npkg_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7786923 entries, 0 to 7786922
Data columns (total 3 columns):
 #   Column     Dtype 
---  ------     ----- 
 0   subject    object
 1   predicate  object
 2   object     object
dtypes: object(3)
memory usage: 178.2+ MB


In [20]:
npkg_df.head()

,subject,predicate,object
0,clo:0035452,rdfs:subClassOf,clo:0000374
1,ensembl:ENST00000256362,rdfs:subClassOf,so:0000673
2,ensembl:ENST00000256362,ro:0002510,ncbigene:55237
3,ensembl:ENST00000256362,rdfs:subClassOf,so:0001217
4,ensembl:ENST00000256362,ro:0002513,pr:q9h8y1


In [21]:
npkg_hashed.get_number_of_edges()

7786923

In [22]:
##workaround till grape dump with weights is fixed
edge_type_mapping = {}
for i in range(len(npkg_df.index)):
    subj = npkg_df.at[i, 'subject']
    pred = npkg_df.at[i, 'predicate']
    obj = npkg_df.at[i, 'object']
    edge_type_mapping[(subj, obj)] = pred
len(edge_type_mapping)

7786923

In [23]:
###dump all edges to file for embeddings
npkg_hashed.dump_edges(
    DIR_PATH+'npkg/train_with_weights.csv',
    separator=',',
    header = False,
    sources_column_number = 0,
    destinations_column_number = 1,
    weights_column_number = 2,
    numeric_node_ids = False,
    directed = True
)

In [24]:
##N=5606586 (70%)
train_sub.dump_edges(
    DIR_PATH+'train_with_weights.csv',
    separator=',',
    header = False,
    sources_column_number = 0,
    destinations_column_number = 1,
    numeric_node_ids = False,
    weights_column_number = 2,
    directed = True
)

In [25]:
##N=622953 (10%)
valid.dump_edges(
    DIR_PATH+'valid_with_weights.csv',
    separator=',',
    header = False,
    sources_column_number = 0,
    destinations_column_number = 1,
    numeric_node_ids = False,
    weights_column_number = 2,
    directed = True
)

In [26]:
##N=1557384 (20%)
test.dump_edges(
    DIR_PATH+'test_with_weights.csv',
    separator=',',
    header = False,
    sources_column_number = 0,
    destinations_column_number = 1,
    numeric_node_ids = False,
    weights_column_number = 2,
    directed = True
)

In [ ]:
###Load data and add edge types to CSV file

In [28]:
npkg_df = pd.read_csv(DIR_PATH+'npkg/train_with_weights.csv', header=None, names=['subject', 'object', 'weight'])
npkg_df.head()

,subject,object,weight
0,clo:0035452,clo:0000374,1021.639830
1,ensembl:ENST00000256362,so:0000673,5488.827600
2,ensembl:ENST00000256362,ncbigene:55237,2.009183
3,ensembl:ENST00000256362,so:0001217,3645.212200
4,ensembl:ENST00000256362,pr:q9h8y1,2.062922


In [31]:
%%time
npkg_df['predicate'] = pd.NA
for i in range(len(npkg_df.index)):
    subj = npkg_df.at[i, 'subject']
    obj = npkg_df.at[i, 'object']
    if (subj, obj) in edge_type_mapping:
        npkg_df.at[i, 'predicate'] = edge_type_mapping[(subj, obj)]
npkg_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7786923 entries, 0 to 7786922
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   subject    object 
 1   object     object 
 2   weight     float64
 3   predicate  object 
dtypes: float64(1), object(3)
memory usage: 237.6+ MB


In [32]:
npkg_df.head()

,subject,object,weight,predicate
0,clo:0035452,clo:0000374,1021.639830,rdfs:subClassOf
1,ensembl:ENST00000256362,so:0000673,5488.827600,rdfs:subClassOf
2,ensembl:ENST00000256362,ncbigene:55237,2.009183,ro:0002510
3,ensembl:ENST00000256362,so:0001217,3645.212200,rdfs:subClassOf
4,ensembl:ENST00000256362,pr:q9h8y1,2.062922,ro:0002513


In [34]:
npkg_df = npkg_df[['subject', 'predicate', 'object', 'weight']]
npkg_df.head()

,subject,predicate,object,weight
0,clo:0035452,rdfs:subClassOf,clo:0000374,1021.639830
1,ensembl:ENST00000256362,rdfs:subClassOf,so:0000673,5488.827600
2,ensembl:ENST00000256362,ro:0002510,ncbigene:55237,2.009183
3,ensembl:ENST00000256362,rdfs:subClassOf,so:0001217,3645.212200
4,ensembl:ENST00000256362,ro:0002513,pr:q9h8y1,2.062922


In [35]:
npkg_df.to_csv(DIR_PATH+'npkg/train_with_weights.csv', header=False, index=False)

In [36]:
train_df = pd.read_csv(DIR_PATH+'train_with_weights.csv', header=None, names=['subject', 'object', 'weight'])
valid_df = pd.read_csv(DIR_PATH+'valid_with_weights.csv', header=None, names=['subject', 'object', 'weight'])
test_df = pd.read_csv(DIR_PATH+'test_with_weights.csv', header=None, names=['subject', 'object', 'weight'])

In [37]:
def add_edge_type(df):
    df['predicate'] = pd.NA
    for i in range(len(df.index)):
        subj = df.at[i, 'subject']
        obj = df.at[i, 'object']
        if (subj, obj) in edge_type_mapping:
            df.at[i, 'predicate'] = edge_type_mapping[(subj, obj)]
    return df

In [38]:
train_df = add_edge_type(train_df)
valid_df = add_edge_type(valid_df)
test_df = add_edge_type(test_df)

In [40]:
train_df = train_df[['subject', 'predicate', 'object', 'weight']]
valid_df = valid_df[['subject', 'predicate', 'object', 'weight']]
test_df = test_df[['subject', 'predicate', 'object', 'weight']]

In [41]:
train_df.head()

,subject,predicate,object,weight
0,clo:0035452,rdfs:subClassOf,clo:0000374,1021.639830
1,ensembl:ENST00000256362,rdfs:subClassOf,so:0000673,5488.827600
2,ensembl:ENST00000256362,ro:0002510,ncbigene:55237,2.009183
3,ensembl:ENST00000256362,rdfs:subClassOf,so:0001217,3645.212200
4,dbsnp:rs1057517106,rdfs:subClassOf,so:0001483,20.153488


In [42]:
train_df.to_csv(DIR_PATH+'train_with_weights.csv', header=False, index=False)
valid_df.to_csv(DIR_PATH+'valid_with_weights.csv', header=False, index=False)
test_df.to_csv(DIR_PATH+'test_with_weights.csv', header=False, index=False)